In [1]:
import subprocess, sys, torch

tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"

subprocess.run([sys.executable,"-m","pip","install",
    "pyg_lib","torch_scatter","torch_sparse",
    "torch_cluster","torch_spline_conv","-f",url,"-q"], check=False)
subprocess.run([sys.executable,"-m","pip","install","torch_geometric","-q"])
print("✔ PyG installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.5 MB/s eta 0:00:00
✔ PyG installed


In [2]:
import shutil, os

CODE = "/kaggle/input/datasets/jerinanan/pgcn-eeg-code"
for f in os.listdir(CODE):
    if f.endswith(".py"):
        shutil.copy(f"{CODE}/{f}", f"/kaggle/working/{f}")
        print(f"  Copied: {f}")

  Copied: kaggle_notebooks.py
  Copied: preprocess_eeg.py
  Copied: pgcn_model.py
  Copied: pgcn_experiment.py
  Copied: preprocess_fusion.py
  Copied: pgcn_statistics_plots.py
  Copied: pgcn_train.py
  Copied: pgcn_core.py


In [3]:
with open("/kaggle/working/pgcn_core.py", "r") as f:
    code = f.read()

code = code.replace(
    "            w=self.cw[targets]",
    "            w=self.cw.to(targets.device)[targets]"
)

with open("/kaggle/working/pgcn_core.py", "w") as f:
    f.write(code)

print("✔ Fixed")

✔ Fixed


In [4]:
import os, json

PRE = "/kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed"
subjects = sorted([d for d in os.listdir(PRE) if d.startswith("s")])
print(f"✔ {len(subjects)} subjects found")

# Check one subject
m = json.load(open(f"{PRE}/s01/meta.json"))
print(f"  s01 windows : {m['total_windows']}")
print(f"  train trials: {len(m['train_trials'])}")
print(f"  val trials  : {len(m['val_trials'])}")
print(f"  test trials : {len(m['test_trials'])}")

✔ 32 subjects found
  s01 windows : 1160
  train trials: 28
  val trials  : 6
  test trials : 6


In [10]:
import sys, shutil, os
sys.path.insert(0, "/kaggle/working")

# Copy code files
for f in os.listdir("/kaggle/input/datasets/jerinanan/pgcn-eeg-code"):
    if f.endswith(".py"):
        shutil.copy(f"/kaggle/input/datasets/jerinanan/pgcn-eeg-code/{f}",
                    f"/kaggle/working/{f}")

from pgcn_experiment import run_experiment, print_summary
# ── Run 2: EEG-Only LOSO ──
store = run_experiment(
    data_dir = "/kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed",
    out_dir  = "/kaggle/working/results_eeg/loso_none",
    mode="loso", fusion="none", ablation="none", force=False,
)
print_summary(store, out_dir="/kaggle/working/results_eeg/loso_none/aggregate")

  mode=loso  fusion=none  ablation=none
  data_dir : /kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed
  out_dir  : /kaggle/working/results_eeg/loso_none

  loso  fusion=none  ablation=none  (32 test folds)
  [SKIP] fold s01
  [SKIP] fold s02
  [SKIP] fold s03
  [SKIP] fold s04
  [SKIP] fold s05
  [SKIP] fold s06
  [SKIP] fold s07

───────────────────────────────────────────────────────
  fold test=s08  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  ]    1160 windows  (1 subjects)
  ep  1/100  tr=5.515  val=0.663  aro=0.725  mean=0.731
  ep 10/100  tr=5.229  val=0.631  aro=0.694  mean=0.715
  Early stop ep18  best=0.7480
  TEST  val=0.7148  aro=0.7302  dom=0.7692  lik=0.9189  quad=0.1296

───────────────────────────────────────────────────────
  fold test=s09  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  

KeyboardInterrupt: 

In [12]:
import shutil, os, json, numpy as np

# Zip everything
shutil.make_archive("/kaggle/working/loso_partial", "zip",
                    "/kaggle/working", "results_eeg")
print(f"✔ Zipped: {os.path.getsize('/kaggle/working/loso_partial.zip')/1e6:.1f} MB")

# Check which folds are done
OUT = "/kaggle/working/results_eeg/loso_none"
done = []
if os.path.exists(OUT):
    for fd in sorted(os.scandir(OUT), key=lambda x: x.name):
        if fd.name.startswith("fold_") and \
           os.path.exists(f"{fd.path}/metrics.json"):
            done.append(fd.name.replace("fold_",""))

remaining = [f"s{i:02d}" for i in range(1,33) if f"s{i:02d}" not in done]
print(f"\nCompleted : {len(done)}/32  → {done}")
print(f"Remaining : {len(remaining)}/32  → {remaining}")

✔ Zipped: 4.9 MB

Completed : 8/32  → ['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08']
Remaining : 24/32  → ['s09', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32']
